# Model Evaluation & Validation

Choosing the right metric, avoiding leakage in evaluation,
calibrating probabilities, and setting decision thresholds from business costs.

## Setup

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "figure.dpi":        130,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         True,
    "grid.alpha":        0.3,
    "axes.titlesize":    13,
    "axes.labelsize":    11,
    "xtick.labelsize":   10,
    "ytick.labelsize":   10,
    "legend.fontsize":   10,
    "font.family":       "sans-serif",
    "lines.linewidth":   2.2,
    "patch.edgecolor":   "none",
})
C = ["#2563EB","#DC2626","#16A34A","#D97706","#7C3AED","#0891B2","#DB2777"]
print("Setup complete")

## 1. ROC-AUC vs PR-AUC — why metric choice matters for imbalance

ROC-AUC looks great even for a useless classifier on imbalanced data
because it counts true negatives. PR-AUC is brutally honest.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, precision_recall_curve,
                              roc_auc_score, average_precision_score)

rng = np.random.default_rng(42)
X, y = make_classification(n_samples=12_000, n_features=12, n_informative=7,
                            weights=[0.97, 0.03], random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

models = {
    "Logistic Reg": LogisticRegression(class_weight="balanced", max_iter=500),
    "Random Forest": RandomForestClassifier(100, class_weight="balanced", random_state=42),
    "GBM":          GradientBoostingClassifier(100, random_state=42),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
naive_ap = y_te.mean()

for (name, model), col in zip(models.items(), C):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:,1]
    fpr, tpr, _ = roc_curve(y_te, proba)
    pre, rec, _ = precision_recall_curve(y_te, proba)
    roc = roc_auc_score(y_te, proba)
    ap  = average_precision_score(y_te, proba)
    axes[0].plot(fpr, tpr, color=col, label=f"{name} (AUC={roc:.3f})")
    axes[1].plot(rec, pre, color=col, label=f"{name} (AP={ap:.3f})")

axes[0].plot([0,1],[0,1],"--", color="grey", lw=1.2, label="Random")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
            title="ROC curve  (3% positive rate)
All models look decent here")
axes[0].legend(fontsize=9)

axes[1].axhline(naive_ap, color="grey", lw=1.5, linestyle="--",
                label=f"No-skill baseline ({naive_ap:.3f})")
axes[1].set(xlabel="Recall", ylabel="Precision",
            title="PR curve  (same data, same models)
Now quality differences are visible")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()
print(f"Positive rate: {y_te.mean():.1%}  ->  use PR-AUC for imbalanced tasks")

## 2. Calibration curves

A model predicting P=0.7 should be right ~70% of the time.
GBMs are typically overconfident (sigmoid-shaped calibration curve);
Logistic Regression is usually well-calibrated.

In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=8000, n_features=10, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

gbm     = GradientBoostingClassifier(200, random_state=42).fit(X_tr, y_tr)
gbm_cal = CalibratedClassifierCV(
    GradientBoostingClassifier(200, random_state=42), method="isotonic", cv=5
).fit(X_tr, y_tr)
lr      = LogisticRegression(max_iter=500).fit(X_tr, y_tr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for model, name, col, ls in [
    (gbm,     "GBM (raw)",        C[1], "--"),
    (gbm_cal, "GBM (calibrated)", C[0], "-"),
    (lr,      "Logistic Reg",     C[2], "-"),
]:
    frac_pos, mean_pred = calibration_curve(y_te, model.predict_proba(X_te)[:,1], n_bins=12)
    ax1.plot(mean_pred, frac_pos, "o-", color=col, ls=ls, lw=2, label=name)

ax1.plot([0,1],[0,1], "--", color="grey", lw=1.2, label="Perfect calibration")
ax1.set(xlabel="Mean predicted probability", ylabel="Fraction of positives",
        title="Calibration curves
(closer to diagonal = better calibrated)")
ax1.legend()

# Histogram of predicted probabilities
for model, name, col in [
    (gbm,     "GBM (raw)",   C[1]),
    (gbm_cal, "GBM (calib)", C[0]),
    (lr,      "LR",          C[2]),
]:
    proba = model.predict_proba(X_te)[:,1]
    ax2.hist(proba, bins=40, alpha=0.55, color=col, label=name, density=True)

ax2.set(xlabel="Predicted probability", ylabel="Density",
        title="Predicted probability distributions
(GBM raw is overconfident near 0 and 1)")
ax2.legend()
plt.tight_layout()
plt.show()

## 3. Cost-sensitive thresholds

The default threshold (0.5) is optimal only when FP and FN have equal costs.
In cancer screening (FN = missed cancer >> FP = unnecessary biopsy), lower thresholds are better.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, stratify=y, test_size=0.25, random_state=42)
pipe = Pipeline([("s", StandardScaler()), ("m", LogisticRegression(max_iter=1000))])
pipe.fit(X_tr, y_tr)
proba = pipe.predict_proba(X_te)[:,1]

cost_fp, cost_fn = 200, 8_000   # GBP: biopsy vs missed cancer
thresholds = np.arange(0.05, 0.96, 0.02)
costs = []
precs, recs = [], []
for t in thresholds:
    pred = (proba >= t).astype(int)
    fp = ((pred==1)&(y_te==0)).sum()
    fn = ((pred==0)&(y_te==1)).sum()
    costs.append(fp*cost_fp + fn*cost_fn)
    precs.append(precision_score(y_te, pred, zero_division=0))
    recs.append(recall_score(y_te, pred))

best_t = thresholds[np.argmin(costs)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(thresholds, costs, color=C[0], lw=2)
ax.axvline(0.5,   color="grey", lw=1.5, linestyle="--", label="Default t=0.50")
ax.axvline(best_t, color=C[1], lw=2,   linestyle="--",
           label=f"Optimal t={best_t:.2f} (min cost)")
ax.set(xlabel="Decision threshold", ylabel="Total expected cost (GBP)",
       title="Cost vs threshold
(FP cost=£200, FN cost=£8,000)")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x,_: f"£{int(x):,}"))

ax = axes[1]
ax.plot(thresholds, precs, color=C[2], lw=2, label="Precision")
ax.plot(thresholds, recs,  color=C[3], lw=2, label="Recall")
ax.axvline(best_t, color=C[1], lw=2, linestyle="--",
           label=f"Optimal t={best_t:.2f}")
ax.set(xlabel="Threshold", ylabel="Score", title="Precision vs Recall
vs threshold")
ax.legend(fontsize=9)

ax = axes[2]
ax.scatter(recs, precs, c=thresholds, cmap="viridis_r", s=40)
sc = ax.scatter(
    [recs[np.argmin(costs)]], [precs[np.argmin(costs)]],
    marker="*", color=C[1], s=250, zorder=5, label=f"Optimal t={best_t:.2f}"
)
ax.set(xlabel="Recall", ylabel="Precision", title="PR curve (color = threshold)")
ax.legend(fontsize=9)
plt.colorbar(
    plt.cm.ScalarMappable(cmap="viridis_r",
                          norm=mpl.colors.Normalize(thresholds.min(), thresholds.max())),
    ax=ax, label="Threshold"
)
plt.tight_layout()
plt.show()
print(f"Default threshold (0.50) cost: £{costs[np.argmin(np.abs(thresholds-0.5))]:,}")
print(f"Optimal threshold ({best_t:.2f}) cost: £{int(min(costs)):,}")
print(f"Savings: £{costs[np.argmin(np.abs(thresholds-0.5))] - min(costs):,.0f}")